# Library

In [1]:
devtools::install_github("shadowdeng1994/LEAP")

── R CMD build ─────────────────────────────────────────────────────────────────
* checking for file ‘/tmp/Rtmp0GZ0oD/remotes2786524220a7d5/shadowdeng1994-LEAP-aba00c6/DESCRIPTION’ ... OK
* preparing ‘LEAP’:
* checking DESCRIPTION meta-information ... OK
* checking for LF line-endings in source and make files and shell scripts
* checking for empty or unneeded directories
* building ‘LEAP_0.0.0.9000.tar.gz’



Installing package into ‘/data/user/dsj/software/R’
(as ‘lib’ is unspecified)



In [2]:
library(LEAP)
library(TarCA.beta)
library(parallel)
library(Seurat)

Loading required package: dplyr


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Loading required package: tidyr

Loading required package: tibble

Loading required package: ggtree

ggtree v3.14.0 Learn more at https://yulab-smu.top/contribution-tree-data/

Please cite:

Shuangbin Xu, Lin Li, Xiao Luo, Meijun Chen, Wenli Tang, Li Zhan, Zehan
Dai, Tommy T. Lam, Yi Guan, Guangchuang Yu. Ggtree: A serialized data
object for visualization of a phylogenetic tree and annotation data.
iMeta 2022, 1(4):e56. doi:10.1002/imt2.56


Attaching package: ‘ggtree’


The following object is masked from ‘package:tidyr’:

    expand


Warning message:
“package ‘Seurat’ was built under R version 4.5.1”
Loading required package: SeuratObject

Loading required package: sp

‘SeuratObject’ was built under R 4.4.1 but the current version is
4.4.2; it is recome

# Load in demo data

In [3]:
load("DemoData/Query.RData",verbose=T)
load("DemoData/Reference.RData",verbose=T)

Loading objects:
  Var.Tree
  Var.Ann
  Var.Expression
Loading objects:
  Data.Ref_Expression
  Data.Ref_metadata


# Processing Query data

## Create TarCA object

In [4]:
Data.TarCA <- CreateTarCAObject(Var.Tree,Var.Ann)

===> Checking input files.

===> Converting to ExTree.

===> Adding AllDescendants.

===> Adding MonoClades.

===> Estimating Np.



## Extract LUGs and reconstruct internal matrix

In [5]:
Var.InternalMatrix <- GetInternalMatrix(Data.TarCA,Var.Expression,Threads=90,SaveObject = TRUE)

[Mon Jun  1 02:19:38 2026] ==> Extracting LUG.
[Mon Jun  1 02:19:38 2026] --> Running LUG estimator.


## Running LUG detector completed.



[Mon Jun  1 02:20:01 2026] --> Shuffling ( 1000 times).
[Mon Jun  1 02:20:01 2026] --> Total:  29 ( 1 / 10 )


	Loading: LEAPOutput//Shuffle//Total_29.rds



[Mon Jun  1 02:20:01 2026] --> Total:  30 ( 2 / 10 )


...

[Mon Jun  1 02:20:01 2026] --> Total:  37 ( 9 / 10 )


	Loading: LEAPOutput//Shuffle//Total_37.rds



[Mon Jun  1 02:20:01 2026] --> Total:  38 ( 10 / 10 )


	Loading: LEAPOutput//Shuffle//Total_38.rds



[Mon Jun  1 02:20:01 2026] --> Extracting high confident LUGs.
[Mon Jun  1 02:20:12 2026] ==> Converting to PhyloDepth.
[Mon Jun  1 02:20:47 2026] ==> Inferring internal state.
[Mon Jun  1 02:21:00 2026] ==> Saving to  LEAPOutput/ .


# Processing Reference data

## Binarized Reference matrix

In [6]:
# Filter overlapped LUGs
tmp.overlapLUG <- intersect(rownames(Var.InternalMatrix),rownames(Data.Ref_Expression))

# Binarizing with top expressing cells
Var.ReferenceMatrix <- GetReferenceMatrix(Data.Ref_Expression[tmp.overlapLUG,],Data.Ref_metadata,GroupBy="embryo.time.bin",Threads = 90,SaveObject = TRUE)

[Mon Jun  1 02:21:00 2026] --> Group:  < 100
[Mon Jun  1 02:21:20 2026] --> Group:  > 650
[Mon Jun  1 02:21:32 2026] --> Group:  100-130
[Mon Jun  1 02:21:47 2026] --> Group:  130-170
[Mon Jun  1 02:22:03 2026] --> Group:  170-210
[Mon Jun  1 02:22:29 2026] --> Group:  210-270
[Mon Jun  1 02:23:04 2026] --> Group:  270-330
[Mon Jun  1 02:23:45 2026] --> Group:  330-390
[Mon Jun  1 02:24:20 2026] --> Group:  390-450
[Mon Jun  1 02:24:55 2026] --> Group:  450-510
[Mon Jun  1 02:25:32 2026] --> Group:  510-580
[Mon Jun  1 02:25:54 2026] --> Group:  580-650
[Mon Jun  1 02:26:18 2026] ==> Saving to  LEAPOutput/ .


# Assigning internal nodes with label transfer

## Define transferred labels

In [7]:
# Set transferred labels
tmp.Label <- 
setNames(
    paste0(Data.Ref_metadata$embryo.time.bin,"|",Data.Ref_metadata$lineage),
    rownames(Data.Ref_metadata)
)

# Order transferred labels along with ReferenceMatrix (VERY IMPORTANT!)
tmp.Label <- tmp.Label[Var.ReferenceMatrix %>% colnames]

## Conduct label transfer

In [8]:
Var.Predictions <- RunLabelTransfer(Var.InternalMatrix,Var.ReferenceMatrix,tmp.Label,SaveObject = TRUE)

[Mon Jun  1 02:26:37 2026] --> Creating Seurat object.


Warning message:
“Data is of class matrix. Coercing to dgCMatrix.”
Warning message:
“Data is of class data.frame. Coercing to dgCMatrix.”


[Mon Jun  1 02:26:43 2026] ==> Processing Seurat object.


Warning message:
“The default method for RunUMAP has changed from calling Python UMAP via reticulate to the R-native UWOT using the cosine metric
To use Python UMAP via reticulate, set umap.method to 'umap-learn' and metric to 'correlation'
This message will be shown once per session”


[Mon Jun  1 02:34:03 2026] ==> Saving Seurat to  LEAPOutput/ .
[Mon Jun  1 02:36:11 2026] ==> Finding transfer anchors.
[Mon Jun  1 02:36:49 2026] ==> Saving Anchor to  LEAPOutput/ .
[Mon Jun  1 02:36:52 2026] ==> Running label transfer.
[Mon Jun  1 02:36:52 2026] ==> Saving Prediction to  LEAPOutput/ .


## Extract PredictedState

In [9]:
Var.PredictedState <- GetPredictedState(Data.TarCA,Var.Predictions,RemoveRoot="Root",SaveObject = TRUE)

[Mon Jun  1 02:36:54 2026] ==> Saving PredictedState to  LEAPOutput/ .
